# ArtExtract Task 2: Painting Similarity (CLIP + FAISS)

**Goal**: retrieve paintings similar to a query painting using deep visual embeddings and fast nearest-neighbor search.

Similarity dimensions modeled:
1. semantic content (objects/faces/pose)
2. composition
3. color/style cues

## 1. Problem Definition

We solve image-to-image retrieval:
- input: one query painting image
- output: top-k similar paintings

Pipeline:
`image -> CLIP embedding -> L2 normalize -> FAISS IndexFlatIP -> nearest neighbors`

In [ ]:
import os
import re
import random
import warnings
from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd
import torch
import faiss
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")

## 2. Configuration

Expected files:
- metadata: `nga_data/data/objects.csv`, `nga_data/data/published_images.csv`
- local images: `images/*.jpg` (recommended filename prefix: object id, e.g. `12345.jpg`)

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path('.')
IMAGES_DIR = ROOT / 'images'
NGA_DATA_DIR = ROOT / 'nga_data' / 'data'
OBJECTS_CSV = NGA_DATA_DIR / 'objects.csv'
PUBLISHED_CSV = NGA_DATA_DIR / 'published_images.csv'

FORCE_DEVICE = os.environ.get('FORCE_DEVICE', '').strip().lower()
if FORCE_DEVICE in {'cpu', 'cuda', 'mps'}:
    DEVICE = FORCE_DEVICE
elif torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

print(f'Device: {DEVICE}')
print(f'Images dir exists: {IMAGES_DIR.exists()}')
print(f'Metadata exists: {OBJECTS_CSV.exists() and PUBLISHED_CSV.exists()}')

## 3. Load Metadata and Build Image Manifest

If NGA CSVs are missing, retrieval still runs with `label='unknown'` and metrics are skipped.

In [ ]:
def _col_or_default(df: pd.DataFrame, candidates: List[str], default: str) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    return default

def _object_id_from_filename(path: Path) -> int | None:
    # Accept files like 12345.jpg or 12345_anything.png
    m = re.match(r'^(\d+)', path.stem)
    return int(m.group(1)) if m else None

img_paths = sorted([p for p in IMAGES_DIR.glob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}])
if not img_paths:
    raise RuntimeError(f'No images found under {IMAGES_DIR.resolve()}')

MAX_IMAGES = int(os.environ.get('MAX_IMAGES', '1500'))
if MAX_IMAGES > 0:
    img_paths = img_paths[:MAX_IMAGES]

manifest = pd.DataFrame({
    'path': [str(p) for p in img_paths],
    'objectid': [_object_id_from_filename(p) for p in img_paths],
})

if OBJECTS_CSV.exists() and PUBLISHED_CSV.exists():
    objects = pd.read_csv(OBJECTS_CSV, low_memory=False)
    published = pd.read_csv(PUBLISHED_CSV, low_memory=False)

    pid = _col_or_default(published, ['objectid', 'depictstmsobjectid'], 'objectid')
    if pid != 'objectid':
        published = published.rename(columns={pid: 'objectid'})

    title_col = _col_or_default(objects, ['title'], 'title')
    artist_col = _col_or_default(objects, ['attribution', 'displayname', 'artist'], 'attribution')
    label_col = _col_or_default(objects, ['classification', 'visualbrowsertype', 'school'], 'classification')

    obj_cols = ['objectid']
    for c in [title_col, artist_col, label_col]:
        if c in objects.columns and c not in obj_cols:
            obj_cols.append(c)

    merged = manifest.merge(objects[obj_cols], on='objectid', how='left')
    merged = merged.rename(columns={title_col: 'title', artist_col: 'artist', label_col: 'label'})
else:
    merged = manifest.copy()
    merged['title'] = merged['path'].map(lambda x: Path(x).name)
    merged['artist'] = 'unknown'
    merged['label'] = 'unknown'

merged['label'] = merged['label'].fillna('unknown').astype(str)
merged['title'] = merged['title'].fillna('').astype(str)
merged['artist'] = merged['artist'].fillna('unknown').astype(str)

print(f'Total images: {len(merged):,}')
print('Top labels:')
print(merged['label'].value_counts().head(10).to_string())

## 4. Load CLIP Model

Backend selection:
- primary: `openai/CLIP` package (`import clip`)
- fallback: `open_clip_torch`

In [ ]:
CLIP_BACKEND = None

try:
    import clip
    model, preprocess = clip.load('ViT-B/32', device=DEVICE)
    CLIP_BACKEND = 'openai-clip'
    print('CLIP backend: openai-clip (ViT-B/32)')
except Exception:
    import open_clip
    model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai', device=DEVICE)
    CLIP_BACKEND = 'open-clip'
    print('CLIP backend: open-clip (ViT-B-32, openai weights)')

model.eval()

## 5. Embedding Extraction

Failure handling:
- corrupted images are skipped
- embeddings are always L2-normalized for cosine/IP equivalence

In [ ]:
@torch.inference_mode()
def embed_paths(paths: List[str], batch_size: int = 64) -> Tuple[np.ndarray, List[int]]:
    feats = []
    kept_indices = []

    for start in tqdm(range(0, len(paths), batch_size), desc='Embedding'):
        batch_paths = paths[start:start + batch_size]
        tensors = []
        local_kept = []

        for j, p in enumerate(batch_paths):
            try:
                img = Image.open(p).convert('RGB')
                tensors.append(preprocess(img))
                local_kept.append(start + j)
            except (FileNotFoundError, UnidentifiedImageError, OSError):
                continue

        if not tensors:
            continue

        x = torch.stack(tensors).to(DEVICE)
        y = model.encode_image(x)
        y = y / y.norm(dim=-1, keepdim=True).clamp(min=1e-12)

        feats.append(y.detach().cpu().numpy().astype('float32'))
        kept_indices.extend(local_kept)

    if not feats:
        raise RuntimeError('No embeddings were produced. Check image decoding and CLIP setup.')

    emb = np.vstack(feats)
    return emb, kept_indices

embeddings, kept = embed_paths(merged['path'].tolist(), batch_size=64)
df = merged.iloc[kept].reset_index(drop=True)

faiss.normalize_L2(embeddings)
print(f'Embeddings: {embeddings.shape}')
print(f'Rows kept: {len(df):,} / {len(merged):,}')

## 6. Build FAISS Index

In [ ]:
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)   # cosine after L2 normalization
index.add(embeddings)

print(f'FAISS index type: {type(index).__name__}')
print(f'Indexed vectors: {index.ntotal:,}')

## 7. Query and Visualization

In [ ]:
@torch.inference_mode()
def _embed_single(path: str) -> np.ndarray:
    img = Image.open(path).convert('RGB')
    x = preprocess(img).unsqueeze(0).to(DEVICE)
    q = model.encode_image(x)
    q = q / q.norm(dim=-1, keepdim=True).clamp(min=1e-12)
    q = q.detach().cpu().numpy().astype('float32')
    faiss.normalize_L2(q)
    return q

def search_similar(query_path: str, k: int = 6) -> pd.DataFrame:
    q = _embed_single(query_path)
    scores, idx = index.search(q, k)
    out = df.iloc[idx[0]].copy()
    out.insert(0, 'rank', np.arange(1, len(out) + 1))
    out.insert(1, 'score', scores[0])
    return out[['rank', 'score', 'path', 'title', 'artist', 'label']]

def show_results(query_path: str, k: int = 5) -> None:
    res = search_similar(query_path, k=k + 1)
    res = res[res['path'] != query_path].head(k)

    plt.figure(figsize=(2.8 * (k + 1), 3.2))
    plt.subplot(1, k + 1, 1)
    plt.imshow(Image.open(query_path).convert('RGB'))
    plt.title('Query')
    plt.axis('off')

    for i, (_, row) in enumerate(res.iterrows(), start=2):
        plt.subplot(1, k + 1, i)
        plt.imshow(Image.open(row['path']).convert('RGB'))
        plt.title(f"#{row['rank']}\n{row['score']:.3f}")
        plt.axis('off')

    plt.tight_layout()
    plt.show()

query_path = df.iloc[0]['path']
print(search_similar(query_path, k=6).to_string(index=False))
show_results(query_path, k=5)

## 8. Retrieval Metrics

Ground truth options for relevance:
- same `label` (default)
- same `artist` (alternative)

Reported metrics:
- Precision@K
- Recall@K
- mAP@K
- nDCG@K

In [ ]:
def precision_at_k(relevant_set: set, retrieved: List[int], k: int) -> float:
    top = retrieved[:k]
    if k == 0:
        return 0.0
    return sum(int(i in relevant_set) for i in top) / k

def recall_at_k(relevant_set: set, retrieved: List[int], k: int) -> float:
    if not relevant_set:
        return 0.0
    top = retrieved[:k]
    return sum(int(i in relevant_set) for i in top) / len(relevant_set)

def ap_at_k(relevant_set: set, retrieved: List[int], k: int) -> float:
    if not relevant_set:
        return 0.0
    hits = 0
    ap = 0.0
    for rank, idx in enumerate(retrieved[:k], start=1):
        if idx in relevant_set:
            hits += 1
            ap += hits / rank
    return ap / min(len(relevant_set), k)

def ndcg_at_k(relevant_set: set, retrieved: List[int], k: int) -> float:
    gains = [1.0 if idx in relevant_set else 0.0 for idx in retrieved[:k]]
    dcg = sum(g / np.log2(i + 2) for i, g in enumerate(gains))
    ideal = sorted(gains, reverse=True)
    idcg = sum(g / np.log2(i + 2) for i, g in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0

K_VALUES = [5, 10]
labels = df['label'].astype(str).to_numpy()
valid_mask = (labels != 'unknown')
label_counts = pd.Series(labels).value_counts()
valid_mask &= np.array([label_counts[l] > 1 for l in labels])
valid_queries = np.where(valid_mask)[0]

if len(valid_queries) == 0:
    print('No valid labels for supervised retrieval metrics. Provide NGA metadata with repeated labels.')
else:
    max_k = max(K_VALUES) + 1
    all_scores, all_idx = index.search(embeddings, max_k)

    rows = []
    for qi in valid_queries:
        retrieved = [int(j) for j in all_idx[qi].tolist() if j != qi]
        relevant = set(np.where(labels == labels[qi])[0].tolist())
        relevant.discard(qi)

        row = {'query_index': int(qi), 'label': labels[qi]}
        for k in K_VALUES:
            row[f'P@{k}'] = precision_at_k(relevant, retrieved, k)
            row[f'R@{k}'] = recall_at_k(relevant, retrieved, k)
            row[f'mAP@{k}'] = ap_at_k(relevant, retrieved, k)
            row[f'nDCG@{k}'] = ndcg_at_k(relevant, retrieved, k)
        rows.append(row)

    mdf = pd.DataFrame(rows)
    print(f'Queries evaluated: {len(mdf):,}')
    print(mdf.drop(columns=['query_index', 'label']).mean().round(4).to_string())

## 9. Embedding Visualization (t-SNE)

In [ ]:
N_TSNE = min(1200, len(df))
if N_TSNE >= 10:
    sample_idx = np.random.choice(len(df), size=N_TSNE, replace=False)
    X = embeddings[sample_idx]
    y = df.iloc[sample_idx]['label'].astype(str).to_numpy()

    perplexity = min(30, max(5, (N_TSNE - 1) // 3))
    tsne = TSNE(n_components=2, perplexity=perplexity, random_state=SEED,
                metric='cosine', init='random', learning_rate='auto')
    Z = tsne.fit_transform(X)

    top_labels = pd.Series(y).value_counts().head(8).index.tolist()
    colors = plt.cm.tab10(np.linspace(0, 1, len(top_labels)))

    plt.figure(figsize=(9, 7))
    for c, lab in zip(colors, top_labels):
        m = (y == lab)
        plt.scatter(Z[m, 0], Z[m, 1], s=10, alpha=0.7, color=c, label=lab)
    plt.title('t-SNE of CLIP Embeddings')
    plt.legend(markerscale=2, fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print('Not enough samples for t-SNE plot.')

## 10. Limitations and Next Steps

Limitations:
- CLIP captures semantics well, but may miss fine brushstroke/material cues.
- Labels in NGA metadata can be sparse/inconsistent for strict retrieval evaluation.
- `IndexFlatIP` is exact search; memory scales linearly with dataset size.

Next steps:
1. test stronger encoders (`ViT-L/14`, OpenCLIP variants)
2. use IVF/HNSW FAISS indexes for larger datasets
3. optionally fine-tune on an art-specific corpus